# BF16 GEMM on Colab G4: A100 노트북의 v1~v6 비교

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/tensor2silicon/blob/main/notebooks/triton_gemm_bf16_g4.ipynb)

[원본 A100 노트북](triton_gemm_bf16_a100.ipynb)의 **outer product → Tensor Core → 파이프라인 → 큰 타일 → L2 순서 → autotune** 흐름을 Colab **G4**에서 실행한다. 입력·출력은 BF16, 누산은 FP32다. 커널 본문과 측정 크기는 원본과 같다.

G4 선택 후 실제 장치가 **NVIDIA RTX PRO 6000 Blackwell Server Edition (sm_120)**인지 먼저 확인한다. A100의 HBM 표기를 GDDR7으로 바꾸고, 블록당 공유 메모리 한도에 맞게 v4·v5의 stage 수와 v6 후보를 조정했다. A100에서 측정한 수치는 가져오지 않는다.

| 버전 | 바꾸는 것 | G4 설정 |
|---|---|---|
| v1 | `tl.dot` 없이 outer product | 64×64, CUDA core FP32 FMA |
| v2 | `tl.dot` 타일 | 64×64×32, 4 warps, 1 stage |
| v3 | 파이프라인 깊이 | v2 + 4 stages |
| v4 | 큰 타일 | 128×128×64, 8 warps, **2 stages** |
| v5 | L2 친화 블록 순서 | v4 + `GROUP_SIZE_M=8` |
| v6 | shape별 autotune | G4 공유 메모리 예산 안의 후보 |

**Colab UI**: `런타임 → 런타임 유형 변경 → G4 GPU` 후 위에서부터 실행한다. 현재 Colab의 PyTorch·Triton을 사용한다. CUDA와 Triton이 sm_120을 지원해야 한다.

**Colab CLI** (저장소 루트에서):
```bash
colab new -s triton-gemm-g4 --gpu G4
colab exec -s triton-gemm-g4 -f notebooks/triton_gemm_bf16_g4.ipynb --timeout 900
colab download -s triton-gemm-g4 /content/triton_gemm_bf16_g4_results.json notebooks/triton_gemm_bf16_g4_results.json
colab stop -s triton-gemm-g4
colab sessions
```
CLI가 코드 셀 오류를 출력해도 다음 셀을 실행할 수 있으므로, `triton_gemm_bf16_g4_output.ipynb`의 error 출력과 마지막 `G4_GEMM_RUN_COMPLETE`를 함께 확인한다. 실패한 경우에도 생성한 세션을 종료한다.

참고: [NVIDIA RTX PRO 6000 사양](https://www.nvidia.com/en-us/data-center/rtx-pro-6000-blackwell-server-edition/), [Blackwell 공유 메모리 한도](https://docs.nvidia.com/cuda/archive/12.8.1/blackwell-tuning-guide/index.html), [Triton matmul 튜토리얼](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html).

## 0. 환경 확인

In [ ]:
# @title 02 G4 GEMM
import torch, triton
import triton.language as tl
import triton.testing
from triton.runtime import driver
from IPython.display import display
from datetime import datetime, timezone

RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
assert torch.cuda.is_available(), "런타임 유형을 G4 GPU로 바꾸세요"
p = torch.cuda.get_device_properties(0)
assert "RTX PRO 6000" in p.name and "Blackwell" in p.name and p.major == 12, (
    f"G4 검증용 노트북입니다. 실제 장치: {p.name}, cc={p.major}.{p.minor}"
)
assert torch.cuda.is_bf16_supported(including_emulation=False), "네이티브 BF16이 필요합니다"
MAX_SHARED_MEM = driver.active.utils.get_device_properties(0)["max_shared_mem"]
ENVIRONMENT = dict(requested_gpu="G4", gpu=p.name, memory_gib=p.total_memory / 2**30,
                   compute_capability=f"{p.major}.{p.minor}", sms=p.multi_processor_count,
                   max_shared_mem_bytes=MAX_SHARED_MEM, torch=torch.__version__,
                   cuda=torch.version.cuda, triton=triton.__version__)
print("torch", torch.__version__, "| cuda", torch.version.cuda, "| triton", triton.__version__)
print(p.name, "| SMs", p.multi_processor_count, "| GDDR7", round(p.total_memory / 2**30, 1),
      "GiB | cc", f"{p.major}.{p.minor}", "| max smem/block", MAX_SHARED_MEM, "bytes")
# FP32 정답을 TF32로 축약하지 않고, cuBLAS도 FP32 누산을 사용한다.
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

## 1. 공통 하네스

- 입력: `a ~ N(0,1)`, `b ~ N(0,1)/√K`. 출력 크기가 1 근처에 머물도록 한다.
- 정답: TF32를 끈 `a.float() @ b.float()` (FP32). BF16 반올림을 감안해 `atol = rtol = 2e-2`로 비교한다.
- 시간: `triton.testing.do_bench`의 median (`warmup=100 ms`, `rep=250 ms`). 컴파일·autotune은 제외하며, 측정 사이마다 Triton 하네스가 캐시를 비운다.
- TFLOPs/s = 2·M·N·K / 시간. 기준은 **같은 G4에서 측정한 cuBLAS**(`torch.matmul`)다.
- `PEAK_TFLOPS=None`이 기본이다. 공개 사양의 sparse/dense·클럭 조건을 확인한 경우에만 BF16 **dense** peak를 직접 넣어 `% of dense peak`를 계산한다. A100의 312를 G4에 적용하지 않는다.

In [ ]:
# @title 04 G4 GEMM
import math
import pandas as pd

PEAK_TFLOPS = None           # G4 BF16 dense peak를 확인한 경우에만 지정
ATOL = RTOL = 2e-2


def make_inputs(M, N, K, seed=0):
    g = torch.Generator(device="cuda").manual_seed(seed)
    a = torch.randn(M, K, device="cuda", generator=g).to(torch.bfloat16)
    b = (torch.randn(K, N, device="cuda", generator=g) / math.sqrt(K)).to(torch.bfloat16)
    return a, b


def check(fn, M, N, K):
    a, b = make_inputs(M, N, K, seed=M + 7 * N + 13 * K)
    c = fn(a, b)
    ref = a.float() @ b.float()
    assert c.shape == (M, N) and c.dtype == torch.bfloat16, (c.shape, c.dtype)
    err = (c.float() - ref).abs().max().item()
    ok = torch.isfinite(c).all().item() and torch.allclose(c.float(), ref, atol=ATOL, rtol=RTOL)
    return ok, err


def bench(fn, M, N, K):
    a, b = make_inputs(M, N, K)
    fn(a, b)                                     # 컴파일·autotune은 측정에서 뺀다
    ms = triton.testing.do_bench(lambda: fn(a, b), warmup=100, rep=250, return_mode="median")
    return ms, 2 * M * N * K / (ms * 1e-3) / 1e12


def cublas(a, b):
    return torch.matmul(a, b)

## 2. 버전별 커널

### v1. outer product: `tl.dot` 없이 CUDA core로

프로그램 하나가 출력 타일 `BLOCK_M × BLOCK_N`을 맡는다. k를 **하나씩** 돌면서 A의 열 하나(길이 BLOCK_M)와 B의 행 하나(길이 BLOCK_N)를 읽고, 외적을 누산기에 더한다. SGEMM_CUDA K5의 register blocking과 같은 계산이다.

- `tl.dot`을 쓰지 않으므로 Tensor Core가 아니라 CUDA core의 FP32 FMA로 계산한다.
- A의 열은 메모리에서 K 간격으로 떨어져 있다. 그래서 K1처럼 coalescing되지 않는 접근이 된다.
- 입력 조각을 shared memory에 올려 재사용하는 구조가 없다. k마다 L1/L2에서 다시 읽는다.

In [ ]:
# @title 06 G4 GEMM
@triton.jit
def outer_product_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    mask_m = offs_m < M
    mask_n = offs_n < N
    a_ptrs = a_ptr + offs_m * stride_am          # A[:, 0]
    b_ptrs = b_ptr + offs_n * stride_bn          # B[0, :]

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, K):
        a = tl.load(a_ptrs, mask=mask_m, other=0.0).to(tl.float32)
        b = tl.load(b_ptrs, mask=mask_n, other=0.0).to(tl.float32)
        acc += a[:, None] * b[None, :]           # rank-1 update = FFMA
        a_ptrs += stride_ak
        b_ptrs += stride_bk

    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc.to(tl.bfloat16), mask=mask_m[:, None] & mask_n[None, :])


V1 = dict(BLOCK_M=64, BLOCK_N=64, num_warps=4)


def launch_v1(a, b):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.bfloat16)
    grid = (triton.cdiv(M, V1["BLOCK_M"]), triton.cdiv(N, V1["BLOCK_N"]))
    handle = outer_product_kernel[grid](
        a, b, c, M, N, K, *a.stride(), *b.stride(), *c.stride(), **V1)
    return c, handle


def gemm_v1(a, b):
    return launch_v1(a, b)[0]

### v2~v5. 타일 GEMM 커널 하나, 파라미터만 바꾼다

v2부터 v5까지는 **커널 코드가 하나**다. 바뀌는 것은 launch 파라미터뿐이다. Triton에서 최적화한다는 것은 대부분 이 파라미터를 고르는 일이다.

커널 본문은 4주차 [Tiling](../week4/03-tiling.md)의 블록 알고리즘을 그대로 옮긴 것이다. 출력 타일을 고정하고, K 방향으로 A·B 조각을 바꿔 가며 `tl.dot`으로 누산한다. 경계는 `mask`로 처리한다. 그래서 SGEMM_CUDA runner의 "M<128이면 64 타일" 같은 우회 없이 아무 크기에서나 동작한다.

`tl.dot`이 컴파일러에 넘기는 일은 다음과 같다.

| CUDA에서 직접 하던 일 | 해당 커널 | Triton에서 |
|---|---|---|
| A·B 조각을 shared memory에 올리기 | K3 | `tl.dot` 피연산자를 컴파일러가 smem에 배치 |
| warp/thread별 출력 분할 | K5·K10 | `num_warps`에 맞춰 자동 분할 |
| coalescing, 128-bit 로드 | K2·K6 | 포인터 연속성을 분석해 자동 벡터화 |
| bank conflict 회피 | K7·K8 | smem swizzle 자동 적용 |
| 연산 명령 | FFMA | BF16 Tensor Core 경로 (실제 PTX/SASS는 4절에서 확인) |

**블록 순서(`GROUP_SIZE_M`)**: 1차원 program id를 출력 타일 (pid_m, pid_n)에 대응시키는 방식이다. `GROUP_SIZE_M=1`이면 행 우선으로 타일을 돈다. 8이면 행 8개 묶음 안에서 열 방향으로 지그재그로 돈다. 동시에 실행되는 프로그램들이 같은 A 행 블록과 B 열 블록을 공유하게 되어 L2 적중률이 올라간다. SGEMM_CUDA K4 주석에 적힌 "blockIdx 순서를 바꾸면 30% 차이" 문제와 같은 원리다.

In [ ]:
# @title 08 G4 GEMM
@triton.jit
def gemm_kernel(
    a_ptr, b_ptr, c_ptr, M, N, K,
    stride_am, stride_ak, stride_bk, stride_bn, stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
):
    # program id -> 출력 타일 (pid_m, pid_n). GROUP_SIZE_M=1이면 행 우선 순서.
    pid = tl.program_id(0)
    num_pid_m = tl.cdiv(M, BLOCK_M)
    num_pid_n = tl.cdiv(N, BLOCK_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak
    b_ptrs = b_ptr + offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        k_left = K - k * BLOCK_K
        a = tl.load(a_ptrs, mask=(offs_m[:, None] < M) & (offs_k[None, :] < k_left), other=0.0)
        b = tl.load(b_ptrs, mask=(offs_k[:, None] < k_left) & (offs_n[None, :] < N), other=0.0)
        acc = tl.dot(a, b, acc)                  # bf16 x bf16 -> fp32, Tensor Core
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    tl.store(c_ptrs, acc.to(tl.bfloat16), mask=(offs_m[:, None] < M) & (offs_n[None, :] < N))


def launch_gemm(a, b, cfg, kernel=gemm_kernel):
    M, K = a.shape
    _, N = b.shape
    c = torch.empty((M, N), device=a.device, dtype=torch.bfloat16)
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_M"]) * triton.cdiv(N, meta["BLOCK_N"]),)
    handle = kernel[grid](a, b, c, M, N, K, *a.stride(), *b.stride(), *c.stride(), **cfg)
    return c, handle

| 버전 | BLOCK_M×N×K | num_warps | num_stages | GROUP_SIZE_M | 의도 |
|---|---|---|---|---|---|
| v2 | 64×64×32 | 4 | 1 | 1 | Tensor Core 기본 타일 |
| v3 | 64×64×32 | 4 | 4 | 1 | K 루프 파이프라인 |
| v4 | 128×128×64 | 8 | **2** | 1 | 큰 타일, stage 버퍼 추정 64 KiB |
| v5 | 128×128×64 | 8 | **2** | 8 | 블록 순서로 L2 재사용 |

G4의 sm_120은 블록당 최대 공유 메모리가 **99 KiB**다. A100의 v4·v5는 3 stage 버퍼만 96 KiB이고, 추가 컴파일러 버퍼까지 고려하면 여유가 작다. G4 버전에서는 2 stage로 줄인다. 버퍼 추정식은 `stages × (BM·BK + BK·BN) × 2 bytes`이며 실제 사용량은 `handle.metadata.shared`로 확인한다.

`num_stages`는 K 루프의 소프트웨어 파이프라인 깊이다. stage가 많으면 로드와 연산을 겹칠 기회가 늘지만 공유 메모리 사용량도 커진다. 모든 단계가 항상 더 빨라지는 것은 아니다.

In [ ]:
# @title 10 G4 GEMM
V2 = dict(BLOCK_M=64,  BLOCK_N=64,  BLOCK_K=32, GROUP_SIZE_M=1, num_warps=4, num_stages=1)
V3 = dict(BLOCK_M=64,  BLOCK_N=64,  BLOCK_K=32, GROUP_SIZE_M=1, num_warps=4, num_stages=4)
V4 = dict(BLOCK_M=128, BLOCK_N=128, BLOCK_K=64, GROUP_SIZE_M=1, num_warps=8, num_stages=2)
V5 = dict(V4, GROUP_SIZE_M=8)


def gemm_v2(a, b): return launch_gemm(a, b, V2)[0]
def gemm_v3(a, b): return launch_gemm(a, b, V3)[0]
def gemm_v4(a, b): return launch_gemm(a, b, V4)[0]
def gemm_v5(a, b): return launch_gemm(a, b, V5)[0]

### v6. G4 공유 메모리 예산으로 autotune

원본처럼 같은 `gemm_kernel`을 `@triton.autotune`으로 감싼다. 처음 호출할 때 shape별 후보를 측정하고 가장 빠른 설정을 캐시한다.

A100 후보 중 128×256×64·3 stage (144 KiB), 128×128×64·4 stage (128 KiB)는 G4 한도를 넘는다. 후보를 32 또는 64 K 타일과 2~5 stage로 구성하고, **추정 stage 버퍼 ≤ min(80 KiB, 실제 한도−16 KiB)**인 후보만 사용한다. 이 추정은 컴파일러의 실제 사용량을 보장하지 않으므로 PTX 분석에서 실제 공유 메모리도 확인한다. 16개 후보에 대한 제한된 탐색이며 전역 최적값을 의미하지 않는다.

In [ ]:
# @title 12 G4 GEMM
def _cfg(bm, bn, bk, stages, warps):
    return triton.Config(dict(BLOCK_M=bm, BLOCK_N=bn, BLOCK_K=bk, GROUP_SIZE_M=8),
                         num_stages=stages, num_warps=warps)


def estimated_smem(cfg):
    q = cfg.kwargs
    return cfg.num_stages * (q["BLOCK_M"] * q["BLOCK_K"] + q["BLOCK_K"] * q["BLOCK_N"]) * 2


CANDIDATE_CONFIGS = [
    _cfg(128, 256, 32, 3, 8), _cfg(256, 128, 32, 3, 8),
    _cfg(128, 128, 64, 2, 4), _cfg(128, 128, 64, 2, 8),
    _cfg(128, 128, 32, 4, 4), _cfg(128, 128, 32, 3, 8),
    _cfg(64, 256, 32, 4, 4), _cfg(256, 64, 32, 4, 4),
    _cfg(128, 64, 32, 4, 4), _cfg(64, 128, 32, 4, 4),
    _cfg(64, 128, 64, 3, 4), _cfg(128, 64, 64, 3, 4),
    _cfg(64, 64, 64, 3, 4), _cfg(128, 32, 32, 4, 4),
    _cfg(64, 32, 32, 5, 2), _cfg(32, 64, 32, 5, 2),
]
SMEM_BUDGET = min(80 * 1024, MAX_SHARED_MEM - 16 * 1024)
AUTOTUNE_CONFIGS = [cfg for cfg in CANDIDATE_CONFIGS if estimated_smem(cfg) <= SMEM_BUDGET]
assert AUTOTUNE_CONFIGS
print("autotune candidates:", len(AUTOTUNE_CONFIGS), "| stage-buffer budget:", SMEM_BUDGET)
# 원본과 같은 커널 본문을 재사용한다.
gemm_kernel_autotuned = triton.autotune(configs=AUTOTUNE_CONFIGS, key=["M", "N", "K"])(gemm_kernel)


def gemm_v6(a, b):
    return launch_gemm(a, b, {}, kernel=gemm_kernel_autotuned)[0]


def best_config_v6():
    best = gemm_kernel_autotuned.best_config
    return dict(best.kwargs, num_warps=best.num_warps, num_stages=best.num_stages)

### G4 Blackwell의 명령 경로를 직접 확인한다

G4의 **sm_120**과 데이터센터 Blackwell의 **sm_100**은 같은 타깃이 아니다. 장치 이름만 보고 Hopper의 `wgmma`나 B200의 `tcgen05` 명령을 가정하지 않는다. 이 노트북은 원본의 포인터 기반 `tl.load` + `tl.dot` 커널을 유지한다.

TMA descriptor나 warp specialization을 추가하는 것은 별도 실험이다. 여기서는 `mma.sync`, `ldmatrix`, `cp.async`와 함께 `wgmma`, `tcgen05`, TMA 계열 명령도 찾아서 **실제 생성된 코드**를 확인한다.

## 3. 정확성 검증

정사각 크기와 타일 배수가 아닌 크기를 섞어 확인한다. v6은 shape마다 처음 호출될 때 autotune이 돌기 때문에 이 셀이 조금 오래 걸린다.

In [ ]:
# @title 15 G4 GEMM
VERSIONS = {"v1": gemm_v1, "v2": gemm_v2, "v3": gemm_v3, "v4": gemm_v4,
            "v5": gemm_v5, "v6": gemm_v6, "cuBLAS": cublas}
CHECK_SHAPES = [(512, 512, 512), (1000, 1000, 1000), (4096, 1024, 2048), (333, 777, 129), (31, 67, 19)]

rows = []
for name, fn in VERSIONS.items():
    for M, N, K in CHECK_SHAPES:
        ok, err = check(fn, M, N, K)
        print(f"check {name} {M}x{N}x{K}: {ok}, max_abs_err={err:.5f}", flush=True)
        rows.append(dict(version=name, shape=f"{M}x{N}x{K}", ok=ok, max_abs_err=round(err, 5)))
check_df = pd.DataFrame(rows)
display(check_df.pivot(index="version", columns="shape", values="max_abs_err").loc[list(VERSIONS)])
assert check_df.ok.all(), check_df[~check_df.ok]
print("모든 버전·shape 통과")

## 4. 컴파일러가 실제로 무엇을 만들었나: PTX와 SASS

`kernel[grid](...)`는 컴파일된 커널 핸들을 돌려준다. `handle.asm["ptx"]`에서 핵심 명령의 **정적 개수**(코드에 등장하는 횟수, 실행 횟수 아님)를 센다.

| 명령 | 뜻 | 관련 커널 |
|---|---|---|
| `fma.rn.f32` | CUDA core FP32 FMA | SGEMM_CUDA 전체 |
| `mma.sync` (SASS `HMMA`) | warp 단위 Tensor Core 행렬곱 | 없음 (새로 생긴 것) |
| `ldmatrix` (SASS `LDSM`) | smem → mma fragment 로드 | K5·K10의 regM/regN 로드 |
| `cp.async` (SASS `LDGSTS`) | GMEM → SMEM 비동기 복사 | K11·K12 |
| `ld.global.v4` / `.v2` | 벡터 로드 | K6 float4 |
| `bar.sync` | `__syncthreads()` | 전체 |

In [ ]:
# @title 17 G4 GEMM
import re

PTX_PATTERNS = {
    "fma.rn.f32": r"\bfma\.rn\.f32\b",
    "mma.sync": r"\bmma\.sync\b",
    "ldmatrix": r"\bldmatrix\b",
    "cp.async": r"\bcp\.async\.c[ag]\b",
    "ld.global.v4/v2": r"\bld\.global\S*\.v[24]\b",
    "bar.sync": r"\bbar\.sync\b",
    "wgmma": r"\bwgmma\.",
    "tcgen05": r"\btcgen05\.",
    "TMA load": r"\bcp\.async\.bulk\.tensor\.",
}
SASS_PATTERNS = {"FFMA": r"\bFFMA\b", "HMMA": r"\bHMMA\b", "LDSM": r"\bLDSM\b", "LDGSTS": r"\bLDGSTS\b"}

a, b = make_inputs(1024, 1024, 1024)
gemm_v6(a, b)                                     # autotune 결과 확보
handles = {
    "v1": launch_v1(a, b)[1],
    "v2": launch_gemm(a, b, V2)[1],
    "v3": launch_gemm(a, b, V3)[1],
    "v4": launch_gemm(a, b, V4)[1],
    "v5": launch_gemm(a, b, V5)[1],
    "v6": launch_gemm(a, b, best_config_v6())[1],
}

rows = []
for name, h in handles.items():
    ptx = h.asm["ptx"]
    row = {"version": name, **{k: len(re.findall(p, ptx)) for k, p in PTX_PATTERNS.items()}}
    try:
        sass = h.asm["sass"]
        row.update({k: len(re.findall(p, sass)) for k, p in SASS_PATTERNS.items()})
    except Exception as e:                        # nvdisasm이 없으면 SASS는 건너뜀
        row["sass"] = f"n/a ({type(e).__name__})"
    row["smem_bytes"] = h.metadata.shared
    assert h.metadata.shared <= MAX_SHARED_MEM, (name, h.metadata.shared, MAX_SHARED_MEM)
    row["regs"] = getattr(h, "n_regs", None)
    rows.append(row)
ptx_df = pd.DataFrame(rows).set_index("version")
display(ptx_df)
INSPECTION_V6_CONFIG = best_config_v6()
print("v6 best config @1024³:", INSPECTION_V6_CONFIG)

**읽는 법**

- **v1**: FP32 FMA가 있고 Tensor Core 명령이 없으면 CUDA core 경로다.
- **v2 이후**: `mma.sync`와 `ldmatrix` 등 실제 생성된 Tensor Core 경로를 확인한다.
- **v3 이후**: `cp.async` 개수와 v2의 개수를 비교해 파이프라인에 따른 비동기 복사를 확인한다.
- `wgmma`, `tcgen05`, TMA 개수가 0이면 해당 명령을 쓰지 않은 것이다. Blackwell이라는 이름만으로 활성화를 가정하지 않는다.
- **smem_bytes / regs**: 컴파일된 실제 공유 메모리·레지스터 사용량이다. 위 추정식과 차이가 날 수 있다.

아래에서 TTGIR의 공유 메모리 레이아웃도 출력한다. Triton 버전에 따라 이름과 표현이 달라질 수 있다. SASS가 없으면 오류 종류를 표에 기록하고 PTX 분석은 계속한다.

In [ ]:
# @title 19 G4 GEMM
shared_lines = [line for line in handles["v3"].asm["ttgir"].splitlines()
                if line.startswith("#") and "shared" in line.lower()]
print("\n".join(shared_lines) if shared_lines else "이 Triton 버전은 별도 #shared 선언을 출력하지 않습니다.")

## 5. 벤치마크

SGEMM_CUDA와 같은 정사각 크기(128~4096)에 8192를 더해 측정한다. v1은 큰 크기에서 오래 걸리므로 4096까지만 잰다.

In [ ]:
# @title 21 G4 GEMM
SIZES = [128, 256, 512, 1024, 2048, 4096, 8192]
V1_MAX = 4096

rows = []
BENCH_V6_CONFIGS = {}
for n in SIZES:
    for name, fn in VERSIONS.items():
        if name == "v1" and n > V1_MAX:
            continue
        ms, tflops = bench(fn, n, n, n)
        assert math.isfinite(ms) and ms > 0 and math.isfinite(tflops), (name, n, ms)
        rows.append(dict(size=n, version=name, ms=ms, tflops=tflops))
        if name == "v6":
            BENCH_V6_CONFIGS[str(n)] = best_config_v6()
    print(f"{n:>5} done | v6 {BENCH_V6_CONFIGS[str(n)]}", flush=True)
bench_df = pd.DataFrame(rows)
cub = bench_df[bench_df.version == "cuBLAS"].set_index("size").tflops
bench_df["pct_cublas"] = 100 * bench_df.tflops / bench_df["size"].map(cub)
if PEAK_TFLOPS is not None:
    assert PEAK_TFLOPS > 0
    bench_df["pct_peak"] = 100 * bench_df.tflops / PEAK_TFLOPS

order = list(VERSIONS)
display(bench_df.pivot(index="version", columns="size", values="tflops").loc[order].round(1))

4096³에서 버전별로 정리하면 다음과 같다(SGEMM_CUDA README 표와 같은 형식).

In [ ]:
# @title 23 G4 GEMM
t = bench_df[bench_df["size"] == 4096].set_index("version").loc[order]
columns = ["tflops", "pct_cublas"] + (["pct_peak"] if PEAK_TFLOPS is not None else [])
display(t[columns].round(1).rename(columns={"tflops": "TFLOPs/s", "pct_cublas": "% of cuBLAS",
                                         "pct_peak": "% of dense peak"}))

In [ ]:
# @title 24 G4 GEMM
import matplotlib.pyplot as plt

# 범주색은 고정 순서로 부여하고, cuBLAS는 기준선이라 무채색 점선으로 그린다.
COLORS = {"v1": "#2a78d6", "v2": "#eb6834", "v3": "#1baf7a", "v4": "#eda100",
          "v5": "#e87ba4", "v6": "#008300", "cuBLAS": "#52514e"}

fig, ax = plt.subplots(figsize=(8, 4.8))
for name in order:
    d = bench_df[bench_df.version == name].sort_values("size")
    ax.plot(d["size"], d.tflops, marker="o", markersize=5, linewidth=2, color=COLORS[name],
            linestyle="--" if name == "cuBLAS" else "-", label=name)
if PEAK_TFLOPS is not None:
    ax.axhline(PEAK_TFLOPS, color="#9a9893", linewidth=1, linestyle=":")
    ax.text(SIZES[0], PEAK_TFLOPS, f" BF16 dense peak {PEAK_TFLOPS:g}", va="bottom", fontsize=9, color="#52514e")
ax.set_xscale("log", base=2)
ax.set_xticks(SIZES, [str(s) for s in SIZES])
ax.set_xlabel("M = N = K")
ax.set_ylabel("TFLOPs/s")
ax.set_title("Triton BF16 GEMM on Colab G4: v1 → v6 vs cuBLAS")
ax.grid(axis="y", color="#e4e3df", linewidth=0.8)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(frameon=False, ncol=4, loc="upper left", bbox_to_anchor=(0, -0.15))
fig.tight_layout()
plt.show()

### 5.1. 크기가 커질수록 cuBLAS와 v6이 벌어지는 지점

**이 실행에서는 작은 행렬의 지연시간이 비슷하다가, 큰 행렬에서 v6의 지속 처리량이 cuBLAS보다 낮아지는 패턴이 나타난다.** 가장 유력한 설명은 작은 크기에서 가려져 있던 타일 재사용·로드/연산 파이프라인의 효율 차이가 충분한 작업량에서 드러난다는 것이다. 다만 현재 기록에는 cuBLAS 커널과 하드웨어 성능 카운터가 없으므로, 어느 자원이 주원인인지는 확정할 수 없다.

아래는 [저장된 G4 결과 JSON](triton_gemm_bf16_g4_results.json)의 **2026-10-03 KST 실행**을 분석한 고정 스냅샷이다. RTX PRO 6000 Blackwell Server Edition, 188 SM, PyTorch 2.11.0+cu130, CUDA 13.0, Triton 3.6.0 조건이다. 노트북을 다시 실행하면 위 그래프는 갱신되지만 이 해설의 수치는 자동으로 바뀌지 않는다.

| M=N=K | v6 시간 (µs) | cuBLAS 시간 (µs) | v6 / cuBLAS 처리량 (TFLOP/s) | cuBLAS 대비 v6 처리량 | v6 시간 증가율 |
|---:|---:|---:|---:|---:|---:|
| 128 | 7.552 | 8.064 | 0.6 / 0.5 | 106.78% | -6.35% |
| 256 | 9.504 | 10.112 | 3.5 / 3.3 | 106.40% | -6.01% |
| 512 | 11.648 | 12.288 | 23.0 / 21.8 | 105.49% | -5.21% |
| 1024 | 23.936 | 24.576 | 89.7 / 87.4 | 102.67% | -2.60% |
| 2048 | 73.120 | 71.744 | 235.0 / 239.5 | 98.12% | +1.92% |
| 4096 | 380.256 | 363.584 | 361.4 / 378.0 | 95.62% | +4.59% |
| 8192 | 2833.600 | 2648.032 | 388.0 / 415.2 | 93.45% | +7.01% |

처리량 비율은 $100t_{\mathrm{cuBLAS}}/t_{v6}$, 시간 증가율은 $100(t_{v6}/t_{\mathrm{cuBLAS}}-1)$이다. 따라서 8192에서 **처리량은 6.55% 낮고, 시간은 7.01% 길다**. 2048의 1.92% 시간 차이는 작지만 4096에서는 4.59%, 8192에서는 7.01%로 커진다. v6 자체도 235.0 → 361.4 → 388.0 TFLOP/s로 빨라지므로, 큰 행렬에서 v6의 처리량이 떨어지는 현상은 아니다.

128~1024에서 v6의 시간상 이득은 약 0.51~0.64 µs다. 짧은 커널에서는 호출·실행 준비 비용과 측정 변동의 상대적 비중이 크므로, 이 결과만으로 작은 행렬에서 v6이 항상 우수하다고 일반화하지 않는다. 반면 8192의 차이는 185.57 µs여서 작은 크기에서 보인 고정 비용 차이만으로 설명하기 어렵다. 저장된 값은 각 `do_bench` 호출의 median이며, 독립 실행 간 분산이나 신뢰구간은 없다.

### 5.2. 실제 선택된 타일: 작은 행렬은 병렬성, 큰 행렬은 타일 효율

`benchmark_v6_configs`와 `gemm_kernel`의 grid를 대입하면 다음과 같다. 출력 타일 하나를 CTA (Cooperative Thread Array, CUDA thread block) 하나가 맡고, 해당 CTA가 K 전체를 순회한다.

$$
N_{\mathrm{CTA}}=\lceil M/B_M\rceil\lceil N/B_N\rceil,\qquad
N_{K\mathrm{loop}}=\lceil K/B_K\rceil
$$

| M=N=K | 선택된 BM×BN×BK | warps / stages | CTA 수 | CTA 수 / 188 SM | CTA당 K 루프 횟수 |
|---:|---|---:|---:|---:|---:|
| 128 | 64×32×32 | 2 / 5 | 8 | 0.04 | 4 |
| 256 | 64×64×64 | 4 / 3 | 16 | 0.09 | 4 |
| 512 | 32×64×32 | 2 / 5 | 128 | 0.68 | 16 |
| 1024 | 64×64×64 | 4 / 3 | 256 | 1.36 | 16 |
| 2048 | 128×128×32 | 4 / 4 | 256 | 1.36 | 64 |
| 4096 | 128×128×32 | 4 / 4 | 1024 | 5.45 | 128 |
| 8192 | 128×128×32 | 4 / 4 | 4096 | 21.79 | 256 |

128~512에서는 CTA 수가 SM 수보다 적다. 작은 타일은 더 많은 SM에 일을 나눠 주지만, 큰 타일은 A·B를 더 많이 재사용한다. 1024에서는 64×64 타일, 2048부터는 **128×128×32, 4 warps, 4 stages**가 선택된다. 큰 세 크기에서 autotune이 선택한 설정은 동일하다. 이러한 병렬성과 재사용의 절충은 [NVIDIA GEMM 가이드의 타일 크기 설명](https://docs.nvidia.com/deeplearning/performance/dl-performance-matrix-multiplication/index.html#typical-tile-dimensions-in-cublas-and-performance)과 일치한다.

4096과 8192에는 각각 1024개, 4096개의 CTA가 있어, 128~512처럼 전체 CTA 수가 SM 수보다 적은 상황은 벗어난다. 그러나 **CTA 수 / SM 수는 occupancy가 아니다**. 동시에 상주할 수 있는 CTA·warp 수는 실제 레지스터와 공유 메모리 사용량에 의해 제한된다.

마지막 실행 묶음(wave)에 일부 SM만 일을 하는 효과도 단독 주원인으로 보기 어렵다. 예를 들어 *SM당 CTA 하나가 상주한다고 가정*하면, 마지막 wave 낭비를 포함한 슬롯 활용률은 4096에서 $1024/(188\times6)=90.8\%$, 8192에서 $4096/(188\times22)=99.0\%$로 오히려 좋아진다. 이는 설명용 계산이며 실제 occupancy를 측정한 값은 아니다. 측정 크기는 모두 선택된 BM·BN·BK의 배수이므로, 경계 타일에 빈 원소가 생기는 낭비도 이 추세를 설명하지 못한다.

### 5.3. 가장 유력한 원인: 제한된 타일·파이프라인 탐색의 지속 처리량

v6의 autotune은 **동일한 커널 본문에서 16개 파라미터 조합만** 고른다. `GROUP_SIZE_M=8`은 고정이며, 더 넓은 후보 탐색이나 다른 알고리즘으로의 전환은 없다. cuBLAS는 자체 구현과 shape별 선택을 사용하지만, 이번 기록에는 실제 선택된 cuBLAS 타일·명령이 없다. 따라서 “cuBLAS가 특정 큰 타일이나 TMA를 써서 빨라졌다”는 것은 아직 증명되지 않았다.

선택된 큰 행렬용 타일에서 K 한 단계는 BF16 입력 $2(B_MB_K+B_KB_N)=16$ KiB를 읽고 $2B_MB_NB_K$ FLOP을 수행한다. 출력 저장과 CTA 사이의 캐시 재사용을 제외한 **타일 내부 입력 재사용 기준** 연산 집약도는

$$
I_{\mathrm{tile}}=\frac{2B_MB_NB_K}{2(B_MB_K+B_KB_N)}
=\frac{B_MB_N}{B_M+B_N}=64\ \mathrm{FLOP/byte}
$$

이다. 128×256 타일이면 같은 계산이 약 85.3 FLOP/byte로 늘지만, 누산기와 공유 메모리 부담도 커진다. 이 모양은 이미 v6 후보에 있으나 선택되지 않았다. **타일을 키우는 것만으로 이긴다고 볼 수 없고**, 재사용 이득과 자원 사용량·파이프라인을 함께 맞춰야 한다.

현재 BK=32에서는 CTA당 K 루프가 2048/4096/8192에서 64/128/256회다. 매 단계의 로드, 공유 메모리에서 fragment를 읽는 일, 주소 갱신, 동기화와 연산의 겹침이 지속 처리량에 영향을 준다. BK를 키우면 루프 횟수는 줄지만 단계당 데이터와 자원 부담이 늘고 총 수학 연산량은 그대로다. **루프 횟수가 늘었다는 사실만으로 상대 성능 차이의 원인이 입증되지는 않는다.** 큰 크기에서 이러한 비용을 더 잘 숨기는 구현이 유리하다는 가설이다.

노트북의 stage 버퍼 추정식에 따르면 선택 설정은 64 KiB이고, 128×128×64를 4 stages로 쓰면 128 KiB다. 후자는 이 장치의 실측 블록당 한도 99 KiB와 후보 예산 80 KiB를 넘는다. 이는 BK·타일·stage를 동시에 늘리기 어려운 이유다. 이 수치는 **후보 필터의 추정치**이며 실제 컴파일 사용량이 아니다. 블록당 99 KiB 및 레지스터에 따른 occupancy 제약은 [CUDA 13.0 Blackwell 튜닝 가이드](https://docs.nvidia.com/cuda/archive/13.0.2/blackwell-tuning-guide/index.html#occupancy)에서도 확인할 수 있다.

4절의 v6 PTX/SASS는 **1024³의 64×64×64 설정만** 검사했다. 여기서는 `mma.sync`, `ldmatrix`, `cp.async`가 있고, 레지스터 126개/thread와 공유 메모리 32 KiB가 기록되어 있다. 이 값을 2048~8192의 128×128×32 설정에 적용하면 안 된다. 큰 크기의 실제 레지스터 수, spill, 공유 메모리 사용량과 명령 경로는 별도 수집이 필요하다. 또한 sm_120의 결과를 sm_100의 `tcgen05` 또는 Hopper의 `wgmma` 사용 여부로 설명하지 않는다.

### 5.4. 메모리 재사용도 후보지만, GDDR7 병목으로 단정할 수는 없다

BF16 입력 두 개의 크기는 $4n^2$ bytes이고 출력까지 합치면 $6n^2$ bytes다.

| M=N=K | A+B | A+B+C |
|---:|---:|---:|
| 2048 | 16 MiB | 24 MiB |
| 4096 | 64 MiB | 96 MiB |
| 8192 | 256 MiB | 384 MiB |

큰 행렬에서는 전체 입력을 캐시에 유지하기 어려워질 수 있어, CTA 순서와 부분 타일의 재사용 거리가 중요해진다. 동일한 타일·warps·stages에서 순서만 바꾼 v4→v5는 4096에서 약 **2.2%**, 8192에서 약 **5.2%** 처리량 향상을 보인다. 이는 큰 크기에서 블록 순서가 유효할 수 있다는 이 실행 내부의 단서다. 다만 한 번의 비교이고 L2 hit rate는 측정하지 않았으며, v6은 다른 타일 설정에 이미 `GROUP_SIZE_M=8`을 적용한다. 이 차이를 그대로 v6의 개선 여지로 옮길 수 없다.

각 입력을 한 번만 읽고 출력을 한 번만 쓴다는 이상적 가정에서는 $I_{\mathrm{ideal}}=2n^3/(6n^2)=n/3$ FLOP/byte로, 행렬이 커질수록 연산 집약도는 높아진다. 반면 실제 CTA들은 A·B 일부를 반복 요청하고, L2 재사용 여부에 따라 GDDR7 트래픽이 달라진다. 앞 절의 타일 집약도와 이 이상적 집약도는 서로 다른 재사용 범위를 뜻한다. **배열이 커졌다는 사실만으로 GDDR7 대역폭이 병목이라고 결론내릴 수 없다.** 현재 JSON에는 L2 용량·hit rate·DRAM 트래픽·Tensor Core 활용률이 없다.

측정 방식도 구분해야 한다. [Triton 3.6.0 `do_bench` 구현](https://github.com/triton-lang/triton/blob/v3.6.0/python/triton/testing.py)은 각 표본의 시작 이벤트 전에 캐시 비우기를 수행한다. 따라서 반복 호출 사이의 입력 캐시 유지 효과는 줄지만, **한 GEMM 실행 안에서의 CTA 간 L2 재사용은 여전히 가능**하다. 캐시 비우기 자체는 해당 표본의 시작·종료 이벤트 사이에 포함되지 않는다.

### 5.5. 원인을 확정하려면 무엇을 비교해야 하나

현재 증거가 가장 잘 지지하는 해석은 **작은 행렬의 병렬성·지연시간 제약에서, 큰 행렬의 타일 재사용·파이프라인 효율 차이가 드러나는 구간으로 이동했다**는 것이다. 제한된 v6 탐색이 그 차이를 모두 없애지는 못했다. 원인별 기여도를 나누려면 다음 비교가 필요하다.

| 확인할 가설 | 후속 비교 | 판단 근거 |
|---|---|---|
| 작은 차이가 측정 변동인가 | 2048/4096/8192에서 실행 순서를 교대·무작위화하여 독립 반복, median과 분포 기록 | 약 2~7% 시간 차이의 재현성과 클럭·온도 변화 확인 |
| 호출 비용이 작은 크기에 영향을 주는가 | 출력 버퍼를 미리 할당하고 두 경로 모두 같은 방식으로 측정; CUDA graph 결과는 별도 표로 비교 | 현재 `torch.empty`/`torch.matmul`을 포함한 호출과 GPU 커널 자체의 차이 구분 |
| v6 후보/측정 조건이 제한적인가 | 같은 최종 하네스로 후보별 시간 기록; BK·stages·warps와 GROUP_SIZE_M을 통제해 비교 | 파라미터 하나의 효과와 최종 벤치마크에 맞는 후보 선택 확인 |
| 자원·파이프라인이 제한하는가 | 큰 세 shape의 선택 설정을 직접 컴파일하고, Nsight Compute로 두 경로의 커널명·실제 smem/regs·spill·active warps·Tensor Core 활용·stall 확인 | 레지스터/공유 메모리 제약과 로드·동기화 대기의 실제 존재 확인 |
| L2/DRAM 재사용 차이가 중요한가 | 동일 타일에서 GROUP_SIZE_M만 변경하고 L2 hit rate·DRAM bytes/throughput 비교 | 재사용 개선과 실행시간 감소가 함께 나타나는지 확인 |

특히 autotune의 측정과 최종 `bench`가 완전히 같다고 가정하지 않는다. 이 노트북은 최종 측정에 `warmup=100`, `rep=250`을 명시하지만 autotune에는 별도 측정 함수를 지정하지 않았다. [Triton 3.6.0 autotuner](https://github.com/triton-lang/triton/blob/v3.6.0/python/triton/runtime/autotuner.py)는 기본 driver benchmarker로 후보 커널을 측정하며, 최종 래퍼의 출력 할당도 후보 측정에는 들어가지 않는다. 이는 후보 선택을 재확인할 이유이며 현재 격차의 원인으로 확정한 사항은 아니다.

이 절은 저장된 측정값·선택 설정·커널 코드에 근거한 분석이다. GPU 재실행이나 Nsight 프로파일링을 수행한 결과는 포함하지 않는다.

다음 실험: [NVFP4 G4 노트북](nvfp4_gemm_g4.ipynb)에서 입력 정밀도를 낮췄을 때의 양자화 오차와 BF16 대비 연산 시간을 비교한다.

## 6. 실측 결과와 저장

마지막 셀은 **이번 실행의** GPU·소프트웨어·정확성·PTX/SASS·벤치마크·shape별 autotune 설정을 JSON으로 저장한다. Colab의 파일 패널 또는 CLI `download`로 가져올 수 있다.

v1→v2는 CUDA core에서 Tensor Core로 계산 경로가 바뀐다. v2→v3은 파이프라인, v3→v4는 타일과 stage 수, v4→v5는 블록 순서, v5→v6은 shape별 후보 선택의 효과다. 실제 효과와 크기별 역전은 위 표·그래프에서 읽는다. GPU 클럭·부하에 따라 수치는 달라지며 작은 차이는 반복 측정이 필요하다.

A100 결과와 비교할 때는 GPU뿐 아니라 v4·v5 stage 수, v6 후보, cuBLAS 누산 설정도 다르다는 점을 고려한다.

In [ ]:
# @title 26 G4 GEMM
import json
from pathlib import Path

assert len(check_df) == len(VERSIONS) * len(CHECK_SHAPES) and check_df.ok.all()
expected_bench_rows = len(SIZES) * len(VERSIONS) - sum(n > V1_MAX for n in SIZES)
assert len(bench_df) == expected_bench_rows
assert set(bench_df.version) == set(VERSIONS)
assert len(BENCH_V6_CONFIGS) == len(SIZES)
assert all(h.metadata.shared <= MAX_SHARED_MEM for h in handles.values())
report = dict(environment=ENVIRONMENT, started_utc=RUN_STARTED_UTC,
              completed_utc=datetime.now(timezone.utc).isoformat(),
              settings=dict(atol=ATOL, rtol=RTOL, fp32_reference_tf32=False,
                            cublas_bf16_reduced_precision_reduction=False,
                            benchmark="triton.testing.do_bench median; cache flush between samples",
                            warmup_ms=100, rep_ms=250, peak_tflops=PEAK_TFLOPS,
                            v1_max=V1_MAX, v2=V2, v3=V3, v4=V4, v5=V5,
                            autotune_smem_budget=SMEM_BUDGET,
                            autotune_candidates=[dict(c.kwargs, num_stages=c.num_stages,
                                                      num_warps=c.num_warps) for c in AUTOTUNE_CONFIGS]),
              correctness=check_df.to_dict(orient="records"),
              assembly=ptx_df.reset_index().to_dict(orient="records"),
              inspection_v6_config=INSPECTION_V6_CONFIG,
              benchmarks=bench_df.to_dict(orient="records"),
              benchmark_v6_configs=BENCH_V6_CONFIGS)
result_path = Path("triton_gemm_bf16_g4_results.json")
result_path.write_text(json.dumps(report, indent=2, ensure_ascii=False, allow_nan=False) + "\n")
print(f"G4_GEMM_RUN_COMPLETE | {len(check_df)} correctness checks | {len(bench_df)} benchmark rows")
print("GPU:", ENVIRONMENT["gpu"], "| memory GiB:", round(ENVIRONMENT["memory_gib"], 2))
print("4096³ v6:", round(t.loc["v6", "tflops"], 2), "TFLOPs/s | cuBLAS:",
      round(t.loc["cuBLAS", "tflops"], 2), "TFLOPs/s | ratio:", round(t.loc["v6", "pct_cublas"], 1), "%")
print("JSON:", result_path.resolve())